In [15]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/iamathavan/tamil-hate-speech/dev.csv
/kaggle/input/datasets/iamathavan/tamil-hate-speech/train.csv
/kaggle/input/datasets/iamathavan/tamil-hate-speech/test.csv


In [16]:
!pip install -q peft
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"       # use ONE GPU only (T4 x2 would otherwise split batches across both)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import torch
print("GPU available:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

GPU available: True | Tesla T4


In [17]:
import glob, re, unicodedata, json, time, gc, shutil, math, dataclasses
import numpy as np, pandas as pd

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True) + glob.glob(f"./**/{name}", recursive=True)
    assert hits, f"{name} not found - did you add the dataset to the notebook?"
    return hits[0]
train, dev, test = (pd.read_csv(find(f"{n}.csv")) for n in ["train", "dev", "test"])

def clean_text(t):
    t = unicodedata.normalize("NFC", str(t))
    t = re.sub(r"[\u200b\u200c\u200d\ufeff]", "", t)
    t = t.replace("\\n", " ")
    t = re.sub(r"https?://\S+|www\.\S+", " ", t)
    t = re.sub(r"@\S+", " ", t)
    t = re.sub(r"\b\d{1,2}:\d{2}(:\d{2})?\b", " ", t)
    t = re.sub(r"(.)\1{2,}", r"\1\1", t)
    return re.sub(r"\s+", " ", t.lower()).strip()

def build(df):
    df = df.copy(); df["clean"] = df.text.apply(clean_text); return df[df.clean.str.len() > 0]
tr, dv, te = build(train), build(dev), build(test)
conf = tr.groupby("clean").label.nunique(); conf = conf[conf > 1].index
tr = tr[~tr.clean.isin(conf)].drop_duplicates("clean").reset_index(drop=True)
dv = dv[~dv.clean.isin(tr.clean)].drop_duplicates("clean").reset_index(drop=True)

from sklearn.model_selection import train_test_split
tr_fit, tr_val = train_test_split(tr, test_size=0.1, stratify=tr.label, random_state=42)  # val slice picks best epoch/lr
CW = torch.tensor(len(tr_fit) / (2 * np.bincount(tr_fit.label)), dtype=torch.float)       # balanced class weights
print("fit", len(tr_fit), "| val", len(tr_val), "| dev (final score only)", len(dv), "| test", len(te), "| class weights", CW.tolist())


fit 4014 | val 446 | dev (final score only) 474 | test 1576 | class weights [0.7821512222290039, 1.386049747467041]


In [5]:
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, EarlyStoppingCallback, set_seed)
OUT = "results"; os.makedirs(OUT, exist_ok=True)
CKPT = "/kaggle/temp/ckpt" if os.path.exists("/kaggle/temp") else "ckpt"

class TextDS(Dataset):
    def __init__(self, texts, labels, tok, max_len=128):
        self.enc = tok(list(texts), truncation=True, max_length=max_len)
        self.labels = None if labels is None else list(labels)
    def __len__(self): return len(self.enc["input_ids"])
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        if self.labels is not None: item["labels"] = int(self.labels[i])
        return item

def compute_metrics(p):
    logits, y = p; pr = logits.argmax(-1)
    return {"accuracy": accuracy_score(y, pr), "macro_f1": f1_score(y, pr, average="macro")}

class WeightedTrainer(Trainer):
    def __init__(self, *a, class_weights=None, **k):
        super().__init__(*a, **k); self.cw = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels"); out = model(**inputs)
        w = None if self.cw is None else self.cw.to(out.logits.device)
        loss = torch.nn.functional.cross_entropy(out.logits.float(), labels, weight=w)
        return (loss, out) if return_outputs else loss

def run_model(tag, model_name, force=False, seed=42, epochs=5, lr=2e-5, bs=16, max_len=128,
              patience=2, lora=False, fp16=True):
    """Train ONE model, save metrics+logits to results/<tag>.*, free the GPU. Skips if already done."""
    res_path = f"{OUT}/{tag}.json"
    if os.path.exists(res_path) and not force:
        print("already done, skipping:", tag); return json.load(open(res_path))
    set_seed(seed); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); t0 = time.time()

    tok = AutoTokenizer.from_pretrained(model_name)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    model.config.pad_token_id = tok.pad_token_id; model.config.use_cache = False
    if lora:   # train small adapters only (stable + low memory for big decoder models)
        from peft import LoraConfig, get_peft_model, TaskType
        model = get_peft_model(model, LoraConfig(task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32,
                               lora_dropout=0.1, target_modules=["query_key_value"]))
        model.print_trainable_parameters()

    ds_fit, ds_val, ds_dv = (TextDS(d.clean, d.label, tok, max_len) for d in (tr_fit, tr_val, dv))
    out = f"{CKPT}/{tag}"
    valid = {f.name for f in dataclasses.fields(TrainingArguments)}
    steps = math.ceil(len(ds_fit) / bs) * epochs
    ta = dict(output_dir=out, num_train_epochs=epochs, learning_rate=lr, per_device_train_batch_size=bs,
              per_device_eval_batch_size=64, weight_decay=0.01, warmup_steps=int(0.1 * steps), max_grad_norm=1.0,
              save_strategy="epoch", load_best_model_at_end=True, metric_for_best_model="macro_f1",
              save_total_limit=1, fp16=fp16 and torch.cuda.is_available(), seed=seed, report_to="none",
              label_names=["labels"], dataloader_num_workers=2)
    ta["eval_strategy" if "eval_strategy" in valid else "evaluation_strategy"] = "epoch"
    args = TrainingArguments(**{k: v for k, v in ta.items() if k in valid})
    kw = dict(model=model, args=args, train_dataset=ds_fit, eval_dataset=ds_val, compute_metrics=compute_metrics,
              class_weights=CW, callbacks=[EarlyStoppingCallback(early_stopping_patience=patience)])
    try:    trainer = WeightedTrainer(processing_class=tok, **kw)
    except TypeError: trainer = WeightedTrainer(tokenizer=tok, **kw)
    trainer.train()

    val_f1 = trainer.evaluate()["eval_macro_f1"]
    dl = trainer.predict(ds_dv).predictions
    tl = trainer.predict(TextDS(te.clean, None, tok, max_len)).predictions
    pred = dl.argmax(-1)
    p, r, f, _ = precision_recall_fscore_support(dv.label, pred, average="macro", zero_division=0)
    res = dict(tag=tag, model=model_name, lr=lr, seed=seed, val_F1=float(val_f1), accuracy=float(accuracy_score(dv.label, pred)),
               macro_P=float(p), macro_R=float(r), macro_F1=float(f),
               weighted_F1=float(f1_score(dv.label, pred, average="weighted")),
               hate_F1=float(f1_score(dv.label, pred, pos_label=1)), predicts_both_classes=bool(len(set(pred)) == 2),
               train_minutes=round((time.time() - t0) / 60, 1),
               peak_gpu_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2))
    json.dump(res, open(res_path, "w"), indent=1)
    np.savez(f"{OUT}/{tag}_logits.npz", dev=dl, test=tl)
    print(json.dumps(res, indent=1))
    if not res["predicts_both_classes"]: print("WARNING: model collapsed (predicts one class only) - don't report this run")

    del trainer, model; gc.collect(); torch.cuda.empty_cache(); shutil.rmtree(out, ignore_errors=True)   # free GPU + disk
    return res

In [7]:
r1 = run_model("tamilsbert", "l3cube-pune/tamil-sentence-bert-nli", lr=2e-5, epochs=5, patience=2)

config.json:   0%|          | 0.00/668 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/517 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/6.41M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.508548,0.634266,5,0.688341,0.660840


{
 "tag": "tamilsbert",
 "model": "l3cube-pune/tamil-sentence-bert-nli",
 "lr": 2e-05,
 "seed": 42,
 "val_F1": 0.6608402129254268,
 "accuracy": 0.6856540084388185,
 "macro_P": 0.6477496416626851,
 "macro_R": 0.6590740740740741,
 "macro_F1": 0.6511865662427461,
 "weighted_F1": 0.6914371363240656,
 "hate_F1": 0.5415384615384615,
 "predicts_both_classes": true,
 "train_minutes": 3.4,
 "peak_gpu_gb": 3.93
}


In [8]:
r2a = run_model("muril_lr3e-5", "google/muril-base-cased", lr=3e-5, epochs=6, patience=3)
r2b = run_model("muril_lr1e-5", "google/muril-base-cased", lr=1e-5, epochs=6, patience=3)

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  953MB            

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.547554,0.662564,6,0.661435,0.622642


{
 "tag": "muril_lr1e-5",
 "model": "google/muril-base-cased",
 "lr": 1e-05,
 "seed": 42,
 "val_F1": 0.6226417208783697,
 "accuracy": 0.6729957805907173,
 "macro_P": 0.6102077687443541,
 "macro_R": 0.5979012345679012,
 "macro_F1": 0.6013024013024013,
 "weighted_F1": 0.6633653266564659,
 "hate_F1": 0.43223443223443225,
 "predicts_both_classes": true,
 "train_minutes": 3.7,
 "peak_gpu_gb": 3.93
}


In [10]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [11]:
r3 = run_model("bloom560m_lora", "bigscience/bloom-560m", lr=2e-4, epochs=4, patience=2, lora=True, fp16=False)

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

[transformers] BloomForSequenceClassification LOAD REPORT from: bigscience/bloom-560m
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 1,574,912 || all params: 560,791,552 || trainable%: 0.2808


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,nan,0.639013,0.389877
2,0.000000,nan,0.639013,0.389877
3,0.000000,nan,0.639013,0.389877


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.000000,nan,3,0.639013,0.389877


{
 "tag": "bloom560m_lora",
 "model": "bigscience/bloom-560m",
 "lr": 0.0002,
 "seed": 42,
 "val_F1": 0.3898768809849521,
 "accuracy": 0.6835443037974683,
 "macro_P": 0.34177215189873417,
 "macro_R": 0.5,
 "macro_F1": 0.40601503759398494,
 "weighted_F1": 0.5550585324069668,
 "hate_F1": 0.0,
 "predicts_both_classes": false,
 "train_minutes": 3.8,
 "peak_gpu_gb": 3.02
}


In [12]:
import glob
R = pd.DataFrame([json.load(open(f)) for f in sorted(glob.glob(f"{OUT}/*.json"))])
cols = ["tag", "val_F1", "macro_F1", "macro_P", "macro_R", "accuracy", "hate_F1", "predicts_both_classes", "train_minutes", "peak_gpu_gb"]
print(R[cols].round(4).to_string(index=False))

ok = R[R.predicts_both_classes].copy()
best = ok.sort_values("val_F1", ascending=False).drop_duplicates("model")     # best config per model, chosen on validation
print("\nBest per model (chosen on validation):"); print(best[["model", "tag", "val_F1", "macro_F1"]].round(4).to_string(index=False))

from scipy.special import softmax
L = {t: np.load(f"{OUT}/{t}_logits.npz") for t in best.tag}
if len(L) > 1:
    ens_dev = np.mean([softmax(v["dev"], axis=1) for v in L.values()], axis=0)
    print("Ensemble macro-F1 on dev:", round(f1_score(dv.label, ens_dev.argmax(1), average="macro"), 4))

top = best.sort_values("val_F1", ascending=False).iloc[0].tag
pd.DataFrame({"id": te.id, "label": L[top]["test"].argmax(1)}).to_csv(f"{OUT}/test_predictions.csv", index=False)
best.drop(columns=["tag"]).round(4).to_csv(f"{OUT}/final_transformers.csv", index=False)
shutil.make_archive("results_download", "zip", OUT)
print("Test predictions from:", top, "| download results_download.zip from the Output tab")

           tag  val_F1  macro_F1  macro_P  macro_R  accuracy  hate_F1  predicts_both_classes  train_minutes  peak_gpu_gb
bloom560m_lora  0.3899    0.4060   0.3418   0.5000    0.6835   0.0000                  False            3.8         3.02
  muril_lr1e-5  0.6226    0.6013   0.6102   0.5979    0.6730   0.4322                   True            3.7         3.93
  muril_lr3e-5  0.3899    0.4060   0.3418   0.5000    0.6835   0.0000                  False            2.7         3.93
    tamilsbert  0.6608    0.6512   0.6477   0.6591    0.6857   0.5415                   True            3.4         3.93

Best per model (chosen on validation):
                              model          tag  val_F1  macro_F1
l3cube-pune/tamil-sentence-bert-nli   tamilsbert  0.6608    0.6512
            google/muril-base-cased muril_lr1e-5  0.6226    0.6013
Ensemble macro-F1 on dev: 0.6447
Test predictions from: tamilsbert | download results_download.zip from the Output tab


In [6]:
# ===== MuRIL with a CUSTOM architecture + custom training loop (paste as ONE cell; needs Kaggle cells 1-3 run first) =====
# Differences from the standard "AutoModelForSequenceClassification" MuRIL that collapsed:
#   1) Mean pooling over all tokens (instead of the [CLS]/pooler vector)  -> more stable signal
#   2) Small MLP head (768 -> 256 -> 2) with dropout
#   3) Layer-wise learning-rate decay: top layers learn faster, bottom layers barely move -> stops the pre-trained model being wrecked
#   4) Separate, much larger LR for the fresh head than for the encoder
#   5) Label smoothing + class-weighted loss + gradient clipping + warm-up/linear decay
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np, json, time, gc, os
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoTokenizer, DataCollatorWithPadding, get_linear_schedule_with_warmup, set_seed

class MuRILMeanPool(nn.Module):
    def __init__(self, name, dropout=0.2):
        super().__init__()
        self.enc = AutoModel.from_pretrained(name)
        h = self.enc.config.hidden_size
        self.drop = nn.Dropout(dropout)
        self.head = nn.Sequential(nn.Linear(h, 256), nn.GELU(), nn.Dropout(dropout), nn.Linear(256, 2))
    def forward(self, input_ids, attention_mask):
        hs = self.enc(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        m = attention_mask.unsqueeze(-1).to(hs.dtype)
        pooled = (hs * m).sum(1) / m.sum(1).clamp(min=1e-9)          # mean over real tokens only
        return self.head(self.drop(pooled))

def llrd_param_groups(model, enc_lr, head_lr, decay=0.9, wd=0.01):
    n = model.enc.config.num_hidden_layers
    groups = []
    for pname, p in model.enc.named_parameters():
        if pname.startswith("pooler"): continue                       # pooler is unused here
        depth = 0 if pname.startswith("embeddings") else int(pname.split(".")[2]) + 1 if pname.startswith("encoder.layer.") else n
        no_wd = ("bias" in pname) or ("LayerNorm" in pname)
        groups.append({"params": [p], "lr": enc_lr * decay ** (n - depth), "weight_decay": 0.0 if no_wd else wd})
    groups.append({"params": list(model.head.parameters()), "lr": head_lr, "weight_decay": wd})
    return groups

def run_muril_custom(tag, enc_lr=2e-5, head_lr=1e-3, epochs=8, bs=16, patience=3, max_len=128,
                     seed=42, label_smoothing=0.05, force=False, name="google/muril-base-cased"):
    res_path = f"{OUT}/{tag}.json"
    if os.path.exists(res_path) and not force:
        print("already done, skipping:", tag); return json.load(open(res_path))
    set_seed(seed); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); t0 = time.time()
    dev_ = torch.device("cuda" if torch.cuda.is_available() else "cpu"); amp = dev_.type == "cuda"

    tok = AutoTokenizer.from_pretrained(name)
    coll = DataCollatorWithPadding(tok)
    ds_fit, ds_val, ds_dv = (TextDS(d.clean, d.label, tok, max_len) for d in (tr_fit, tr_val, dv))
    ds_te = TextDS(te.clean, None, tok, max_len)
    dl_fit = DataLoader(ds_fit, batch_size=bs, shuffle=True, collate_fn=coll)

    model = MuRILMeanPool(name).to(dev_)
    opt = torch.optim.AdamW(llrd_param_groups(model, enc_lr, head_lr))
    total = len(dl_fit) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    w = CW.to(dev_)

    @torch.no_grad()
    def predict(ds):
        model.eval(); outs = []
        for b in DataLoader(ds, batch_size=64, shuffle=False, collate_fn=coll):
            with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                lg = model(b["input_ids"].to(dev_), b["attention_mask"].to(dev_))
            outs.append(lg.float().cpu())
        return torch.cat(outs).numpy()

    best_f1, best_state, bad = -1, None, 0
    for ep in range(1, epochs + 1):
        model.train(); run_loss = 0
        for b in dl_fit:
            ids, am, y = b["input_ids"].to(dev_), b["attention_mask"].to(dev_), b["labels"].to(dev_)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                logits = model(ids, am)
            loss = F.cross_entropy(logits.float(), y, weight=w, label_smoothing=label_smoothing)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run_loss += loss.item()
        vp = predict(ds_val).argmax(-1)
        vf1 = f1_score(tr_val.label, vp, average="macro")
        print(f"epoch {ep}: train_loss {run_loss/len(dl_fit):.4f} | val macro-F1 {vf1:.4f} | % predicted hate {vp.mean()*100:.1f}")
        if vf1 > best_f1:
            best_f1, bad = vf1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: print("early stop"); break

    model.load_state_dict(best_state)
    dl_, tl_ = predict(ds_dv), predict(ds_te)
    pred = dl_.argmax(-1)
    p, r, f, _ = precision_recall_fscore_support(dv.label, pred, average="macro", zero_division=0)
    res = dict(tag=tag, model="google/muril-base-cased (mean-pool+MLP+LLRD)", lr=enc_lr, seed=seed, val_F1=float(best_f1),
               accuracy=float(accuracy_score(dv.label, pred)), macro_P=float(p), macro_R=float(r), macro_F1=float(f),
               weighted_F1=float(f1_score(dv.label, pred, average="weighted")),
               hate_F1=float(f1_score(dv.label, pred, pos_label=1)), predicts_both_classes=bool(len(set(pred)) == 2),
               train_minutes=round((time.time() - t0) / 60, 1), peak_gpu_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2))
    json.dump(res, open(res_path, "w"), indent=1)
    np.savez(f"{OUT}/{tag}_logits.npz", dev=dl_, test=tl_)
    print(json.dumps(res, indent=1))
    if not res["predicts_both_classes"]: print("WARNING: collapsed (predicts one class only)")
    del model, opt, best_state; gc.collect(); torch.cuda.empty_cache()
    return res

# ---- run (each ~4-6 min on a T4). Edit the list to try other learning rates. ----
for tag, lr in [("muril_meanpool_lr1e-5", 1e-5), ("muril_meanpool_lr2e-5", 2e-5)]:
    run_muril_custom(tag, enc_lr=lr)

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  953MB            

...

model.safetensors: downloading bytes:           |  0.00B            

epoch 1: train_loss 0.6918 | val macro-F1 0.6008 | % predicted hate 40.1
epoch 2: train_loss 0.6543 | val macro-F1 0.6227 | % predicted hate 44.6
epoch 3: train_loss 0.5928 | val macro-F1 0.6355 | % predicted hate 36.1
epoch 4: train_loss 0.5504 | val macro-F1 0.6481 | % predicted hate 35.2
epoch 5: train_loss 0.5115 | val macro-F1 0.6475 | % predicted hate 39.7
epoch 6: train_loss 0.4772 | val macro-F1 0.6558 | % predicted hate 40.4
epoch 7: train_loss 0.4549 | val macro-F1 0.6500 | % predicted hate 37.2
epoch 8: train_loss 0.4357 | val macro-F1 0.6510 | % predicted hate 36.5
{
 "tag": "muril_meanpool_lr1e-5",
 "model": "google/muril-base-cased (mean-pool+MLP+LLRD)",
 "lr": 1e-05,
 "seed": 42,
 "val_F1": 0.6558110819570921,
 "accuracy": 0.630801687763713,
 "macro_P": 0.604047510278666,
 "macro_R": 0.6171604938271604,
 "macro_F1": 0.6032068729640135,
 "weighted_F1": 0.6416188551651952,
 "hate_F1": 0.498567335243553,
 "predicts_both_classes": true,
 "train_minutes": 6.2,
 "peak_gpu_gb":

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


epoch 1: train_loss 0.6900 | val macro-F1 0.5913 | % predicted hate 40.1
epoch 2: train_loss 0.6520 | val macro-F1 0.6206 | % predicted hate 52.5
epoch 3: train_loss 0.5761 | val macro-F1 0.6364 | % predicted hate 37.7
epoch 4: train_loss 0.5000 | val macro-F1 0.6249 | % predicted hate 36.8
epoch 5: train_loss 0.4356 | val macro-F1 0.6155 | % predicted hate 33.9
epoch 6: train_loss 0.3711 | val macro-F1 0.6290 | % predicted hate 30.5
early stop
{
 "tag": "muril_meanpool_lr2e-5",
 "model": "google/muril-base-cased (mean-pool+MLP+LLRD)",
 "lr": 2e-05,
 "seed": 42,
 "val_F1": 0.6364136977870397,
 "accuracy": 0.689873417721519,
 "macro_P": 0.6468841851932219,
 "macro_R": 0.6532098765432099,
 "macro_F1": 0.6494276873537772,
 "weighted_F1": 0.6931392220369268,
 "hate_F1": 0.5303514376996805,
 "predicts_both_classes": true,
 "train_minutes": 4.4,
 "peak_gpu_gb": 4.46
}


In [14]:
!pip uninstall -y torchao

In [10]:
!pip install -q peft

In [ ]:
# ===== BLOOM-560M with a CUSTOM architecture + custom training loop (paste as ONE cell; needs Kaggle cells 1-3 run first) =====
# Why the standard BLOOM classifier collapsed, and what this changes:
#   - BLOOM's hidden states are very large -> a fresh classifier on top explodes (we saw loss = 76).
#   1) Mean pooling over all real tokens (no dependence on BLOOM's last-token / padding-side behaviour)
#   2) A fresh LayerNorm on the pooled vector  -> rescales the huge activations before the head
#   3) LoRA adapters on the attention layers only (small, stable, ~3 GB GPU)
#   4) Head gets a larger LR (1e-3) than the LoRA adapters (5e-5 / 1e-4)
#   5) fp32 (no fp16: BLOOM can overflow), class weights, label smoothing, grad clipping, warm-up + linear decay
# If you get "ImportError ... torchao" run in a cell:   !pip uninstall -y torchao   (then re-run)
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np, json, time, gc, os
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoTokenizer, DataCollatorWithPadding, get_linear_schedule_with_warmup, set_seed

class BloomMeanPool(nn.Module):
    def __init__(self, name, r=16, dropout=0.2):
        super().__init__()
        from peft import LoraConfig, get_peft_model
        base = AutoModel.from_pretrained(name).float()                # BloomModel (no LM head); force fp32 (checkpoint is stored in fp16)
        base.config.use_cache = False
        self.enc = get_peft_model(base, LoraConfig(r=r, lora_alpha=2 * r, lora_dropout=0.1, bias="none",
                                                   target_modules=["query_key_value"]))
        h = base.config.hidden_size
        self.norm = nn.LayerNorm(h)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Sequential(nn.Linear(h, 256), nn.GELU(), nn.Dropout(dropout), nn.Linear(256, 2))
    def forward(self, input_ids, attention_mask):
        hs = self.enc(input_ids=input_ids, attention_mask=attention_mask, use_cache=False).last_hidden_state.float()
        m = attention_mask.unsqueeze(-1).to(hs.dtype)
        pooled = (hs * m).sum(1) / m.sum(1).clamp(min=1e-9)
        return self.head(self.drop(self.norm(pooled)))

def run_bloom_custom(tag, lora_lr=1e-4, head_lr=1e-3, epochs=8, bs=16, patience=3, max_len=128,
                     seed=42, label_smoothing=0.05, force=False, name="bigscience/bloom-560m"):
    res_path = f"{OUT}/{tag}.json"
    if os.path.exists(res_path) and not force:
        print("already done, skipping:", tag); return json.load(open(res_path))
    set_seed(seed); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); t0 = time.time()
    dev_ = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    coll = DataCollatorWithPadding(tok)
    ds_fit, ds_val, ds_dv = (TextDS(d.clean, d.label, tok, max_len) for d in (tr_fit, tr_val, dv))
    ds_te = TextDS(te.clean, None, tok, max_len)
    dl_fit = DataLoader(ds_fit, batch_size=bs, shuffle=True, collate_fn=coll)

    model = BloomMeanPool(name).to(dev_)
    trainable = {n: p for n, p in model.named_parameters() if p.requires_grad}
    lora_p = [p for n, p in trainable.items() if "lora_" in n]
    other_p = [p for n, p in trainable.items() if "lora_" not in n]
    print(f"trainable params: {sum(p.numel() for p in trainable.values())/1e6:.2f}M (LoRA {sum(p.numel() for p in lora_p)/1e6:.2f}M + head)")
    opt = torch.optim.AdamW([{"params": lora_p, "lr": lora_lr, "weight_decay": 0.0},
                             {"params": other_p, "lr": head_lr, "weight_decay": 0.01}])
    total = len(dl_fit) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
    w = CW.to(dev_)

    @torch.no_grad()
    def predict(ds):
        model.eval(); outs = []
        for b in DataLoader(ds, batch_size=64, shuffle=False, collate_fn=coll):
            outs.append(model(b["input_ids"].to(dev_), b["attention_mask"].to(dev_)).float().cpu())
        return torch.cat(outs).numpy()

    best_f1, best_state, bad = -1, None, 0
    for ep in range(1, epochs + 1):
        model.train(); run_loss = 0
        for b in dl_fit:
            ids, am, y = b["input_ids"].to(dev_), b["attention_mask"].to(dev_), b["labels"].to(dev_)
            opt.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(ids, am), y, weight=w, label_smoothing=label_smoothing)
            loss.backward()
            nn.utils.clip_grad_norm_(trainable.values(), 1.0)
            opt.step(); sched.step()
            run_loss += loss.item()
        vp = predict(ds_val).argmax(-1)
        vf1 = f1_score(tr_val.label, vp, average="macro")
        print(f"epoch {ep}: train_loss {run_loss/len(dl_fit):.4f} | val macro-F1 {vf1:.4f} | % predicted hate {vp.mean()*100:.1f}")
        if vf1 > best_f1:
            best_f1, bad = vf1, 0
            best_state = {n: p.detach().cpu().clone() for n, p in trainable.items()}
        else:
            bad += 1
            if bad >= patience: print("early stop"); break

    with torch.no_grad():
        for n, p in trainable.items(): p.copy_(best_state[n].to(p.device))
    dl_, tl_ = predict(ds_dv), predict(ds_te)
    pred = dl_.argmax(-1)
    p, r, f, _ = precision_recall_fscore_support(dv.label, pred, average="macro", zero_division=0)
    res = dict(tag=tag, model="bigscience/bloom-560m (LoRA+mean-pool)", lr=lora_lr, seed=seed, val_F1=float(best_f1),
               accuracy=float(accuracy_score(dv.label, pred)), macro_P=float(p), macro_R=float(r), macro_F1=float(f),
               weighted_F1=float(f1_score(dv.label, pred, average="weighted")),
               hate_F1=float(f1_score(dv.label, pred, pos_label=1)), predicts_both_classes=bool(len(set(pred)) == 2),
               train_minutes=round((time.time() - t0) / 60, 1), peak_gpu_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2))
    json.dump(res, open(res_path, "w"), indent=1)
    np.savez(f"{OUT}/{tag}_logits.npz", dev=dl_, test=tl_)
    print(json.dumps(res, indent=1))
    if not res["predicts_both_classes"]: print("WARNING: collapsed (predicts one class only)")
    del model, opt, best_state, trainable; gc.collect(); torch.cuda.empty_cache()
    return res

# ---- run (each ~5-8 min on a T4). Edit the list to try other learning rates. ----
for tag, lr in [("bloom_meanpool_lr1e-4", 1e-4), ("bloom_meanpool_lr5e-5", 5e-5)]:
    run_bloom_custom(tag, lora_lr=lr)

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

trainable params: 1.84M (LoRA 1.57M + head)
epoch 1: train_loss 0.7003 | val macro-F1 0.5809 | % predicted hate 53.8
epoch 2: train_loss 0.6706 | val macro-F1 0.5884 | % predicted hate 52.0
epoch 3: train_loss 0.6327 | val macro-F1 0.6372 | % predicted hate 30.7
epoch 4: train_loss 0.5744 | val macro-F1 0.6433 | % predicted hate 50.0
epoch 5: train_loss 0.4931 | val macro-F1 0.6254 | % predicted hate 33.9
epoch 6: train_loss 0.4069 | val macro-F1 0.6344 | % predicted hate 34.5
epoch 7: train_loss 0.3389 | val macro-F1 0.6403 | % predicted hate 37.2
early stop
{
 "tag": "bloom_meanpool_lr1e-4",
 "model": "bigscience/bloom-560m (LoRA+mean-pool)",
 "lr": 0.0001,
 "seed": 42,
 "val_F1": 0.6433316929133859,
 "accuracy": 0.6244725738396625,
 "macro_P": 0.6162039680274051,
 "macro_R": 0.6340123456790123,
 "macro_F1": 0.6077034667460199,
 "weighted_F1": 0.63747718750412,
 "hate_F1": 0.526595744680851,
 "predicts_both_classes": true,
 "train_minutes": 23.1,
 "peak_gpu_gb": 9.48
}


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

trainable params: 1.84M (LoRA 1.57M + head)
